# triton-roofline: Colab runner

1. **Runtime → Change runtime type → T4 GPU**
2. If you forked this repo, change `REPO_URL` below
3. **Runtime → Run all**

Colab already has PyTorch and Triton installed.

## 1. Check the GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch, triton
print('torch', torch.__version__, '| triton', triton.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name())

## 2. Get the code

Clones the first time, pulls after that. Re-run this cell whenever you push from your Mac.

In [ ]:
import os
REPO_URL = "https://github.com/coderaashir/triton-roofline.git"

if not os.path.exists("/content/triton-roofline"):
    !git clone {REPO_URL} /content/triton-roofline
else:
    !git -C /content/triton-roofline pull
%cd /content/triton-roofline
!pip install -q -e ".[dev]"

## 3. Run the tests

In [ ]:
!pytest -q

## 4. Day 1 sanity check

Add two big vectors on the GPU and see the time. A T4 can move at most 320 GB/s; this op moves 12 bytes per element (read a, read b, write c).

In [ ]:
from troof.hardware import T4
from triton.testing import do_bench

n = 50_000_000
a = torch.randn(n, device='cuda')
b = torch.randn(n, device='cuda')

ms = do_bench(lambda: a + b)
gbs = 12 * n / (ms * 1e-3) / 1e9
print(f'{ms:.3f} ms  ->  {gbs:.0f} GB/s  ({gbs / T4.mem_bw_gbs:.0%} of the {T4.mem_bw_gbs:.0f} GB/s spec)')